In [5]:
from __future__ import annotations

import json
import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated, Any

# Pydantic v1 is required for LangChain compatibility
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# Ollama Import
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults


In [8]:
from __future__ import annotations

import json
import re
import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated, Any

# --- Pydantic V1 for LangChain Compatibility ---
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# --- LangChain & Ollama ---
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

# ==========================================
# 0. HELPER: Clean JSON from Markdown
# ==========================================
def clean_json_output(text: str) -> str:
    """Removes markdown code fences to ensure json.loads can parse the output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

# ==========================================
# 1. ROBUST SCHEMAS (Pydantic V1)
# ==========================================

class Task(BaseModel):
    id: int = Field(default=0)
    title: str = Field(default="Section")

    @root_validator(pre=True)
    def map_deepseek_task_keys(cls, values: dict[str, Any]) -> dict[str, Any]:
        mapping = {
            "section_title": "title", "section_goal": "goal",
            "content": "bullets", "target_word_count": "target_words"
        }
        for old_key, new_key in mapping.items():
            if old_key in values:
                values[new_key] = values.pop(old_key)
        return values

    goal: str = Field(..., description="Section goal")
    bullets: List[str] = Field(..., min_length=1)
    target_words: int = Field(..., description="Word count")
    
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False

class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task]

    @root_validator(pre=True)
    def fix_deepseek_plan(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "blog_plan" in values: return values["blog_plan"]
        if "plan" in values and isinstance(values["plan"], list):
            return {
                "blog_title": "DeepSeek Generated Blog",
                "audience": "Developers",
                "tone": "Technical",
                "tasks": values["plan"]
            }
        return values

class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

    @root_validator(pre=True)
    def fix_deepseek_router(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "router_decision" in values:
            values = values["router_decision"]
            
        # 1. Default mode if missing
        if "mode" not in values:
            values["mode"] = "hybrid" if values.get("needs_research") else "closed_book"

        # 2. FORCE needs_research=True if mode is hybrid/open_book
        # This fixes the issue where it skips research despite being in hybrid mode
        if values.get("mode") in ["hybrid", "open_book"]:
            values["needs_research"] = True
            
            # Ensure we have at least one query if none provided
            if not values.get("queries"):
                values["queries"] = ["latest trends and best practices"]

        return values

class State(TypedDict):
    topic: str
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[dict]
    plan: Optional[dict]
    sections: Annotated[List[tuple], operator.add] 
    final: str

# ==========================================
# 2. LLM SETUP
# ==========================================
llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud", 
    temperature=0,
)

# ==========================================
# 3. ROUTER NODE
# ==========================================
ROUTER_SYSTEM = """You are a routing module. Decide if web research is needed.
Return STRICT JSON (no markdown):
{
  "needs_research": boolean,
  "mode": "hybrid", 
  "queries": ["query1", "query2"]
}
Modes:
- closed_book: Concepts only.
- hybrid: Concepts + Examples (Requires Research).
- open_book: News/Trends (Requires Research).
"""

def router_node(state: State) -> dict:
    print(f"--- Router Node (Topic: {state['topic']}) ---")
    response = llm.invoke(
        [
            SystemMessage(content=ROUTER_SYSTEM),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        # The validator logic in RouterDecision will now FORCE research=True for hybrid
        decision = RouterDecision(**data)
        
        print(f"Mode: {decision.mode} | Research Required: {decision.needs_research}")
        return {
            "needs_research": decision.needs_research,
            "mode": decision.mode,
            "queries": decision.queries,
        }
    except Exception as e:
        print(f"Router Error: {e}. Defaulting to Hybrid Research.")
        return {
            "needs_research": True, 
            "mode": "hybrid", 
            "queries": [f"{state['topic']} trends 2025", f"{state['topic']} best practices"]
        }

def route_next(state: State) -> str:
    # This will now correctly route to 'research' because needs_research is enforced
    return "research" if state["needs_research"] else "orchestrator"

# ==========================================
# 4. RESEARCH NODE
# ==========================================
def _tavily_search(query: str, max_results: int = 3) -> List[dict]:
    print(f"  > Searching Tavily: {query}")
    try:
        tool = TavilySearchResults(max_results=max_results)
        results = tool.invoke({"query": query})
        normalized = []
        for r in results or []:
            normalized.append({
                "title": r.get("title", "No Title"),
                "url": r.get("url", ""),
                "snippet": r.get("content", "") or r.get("snippet", ""),
                "published_at": r.get("published_date")
            })
        return normalized
    except Exception as e:
        print(f"    Tavily API Error: {e}")
        return []

RESEARCH_SYSTEM = """Synthesize search results into a JSON object.
Return STRICT JSON (no markdown):
{
  "evidence": [
    { "title": "...", "url": "...", "snippet": "..." }
  ]
}
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", [])[:3]
    if not queries:
        queries = [f"{state['topic']} analysis"]

    raw_results = []
    for q in queries:
        raw_results.extend(_tavily_search(q))

    if not raw_results:
        print("  > No results found.")
        return {"evidence": []}

    print("  > Synthesizing evidence...")
    response = llm.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw Results:\n{str(raw_results)[:10000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        items = data if isinstance(data, list) else data.get("evidence", [])
        
        valid_evidence = []
        seen_urls = set()
        for item in items:
            if item.get("url") and item["url"] not in seen_urls:
                valid_evidence.append(item)
                seen_urls.add(item["url"])
                
        print(f"  > Found {len(valid_evidence)} valid evidence items.")
        return {"evidence": valid_evidence}

    except Exception as e:
        print(f"  > Research Parse Error: {e}")
        return {"evidence": []}

# ==========================================
# 5. ORCHESTRATOR NODE
# ==========================================
ORCH_SYSTEM = """Create a blog plan.
Return STRICT JSON (no markdown):
{
  "blog_title": "...",
  "audience": "...",
  "tone": "...",
  "tasks": [
    { "title": "...", "goal": "...", "bullets": ["..."], "target_words": 200 }
  ]
}
"""

def orchestrator_node(state: State) -> dict:
    print("--- Orchestrator Node ---")
    evidence = state.get("evidence", [])
    
    response = llm.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Evidence: {[e.get('title') for e in evidence[:5]]}"
                )
            ),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        plan = Plan(**data)
        return {"plan": plan.dict()}
    except Exception as e:
        print(f"Plan Parse Error: {e}")
        # Fallback Plan
        fallback_plan = Plan(
            blog_title=f"Guide to {state['topic']}",
            audience="Developers",
            tone="Technical",
            tasks=[
                Task(id=1, title="Overview", goal="Intro", bullets=["Key concept 1", "Key concept 2"], target_words=200)
            ]
        )
        return {"plan": fallback_plan.dict()}

# ==========================================
# 6. FANOUT & WORKER
# ==========================================
def fanout(state: State):
    tasks_with_ids = []
    if state["plan"]:
        plan_data = state["plan"]
        tasks = plan_data.get("tasks", [])
        
        for i, task_data in enumerate(tasks):
            if "id" not in task_data or task_data["id"] == 0:
                task_data["id"] = i + 1
            tasks_with_ids.append(task_data)

    return [
        Send("worker", {
            "task": task,
            "topic": state["topic"],
            "plan": state["plan"],
            "evidence": state.get("evidence", []),
        }) for task in tasks_with_ids
    ]

WORKER_SYSTEM = """You are a technical writer. Write ONE section in Markdown.
Constraints:
- Use 

[Image of X]
 tags to suggest relevant diagrams.
- Cover all bullets.
- Use evidence URLs for citations if available.
- Start with '## Title'.
- Do NOT output JSON. Output raw Markdown.
"""

def worker_node(payload: dict) -> dict:
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = payload.get("evidence", [])
    
    bullets_text = "\n- " + "\n- ".join(task.bullets)
    evidence_text = "\n".join([f"- {e.get('title')} ({e.get('url')})" for e in evidence[:10]])

    print(f"Writing Section: {task.title}")
    
    response = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Section: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Bullets:{bullets_text}\n"
                    f"Evidence:\n{evidence_text}\n"
                )
            ),
        ]
    )
    return {"sections": [(task.id, response.content.strip())]}

# ==========================================
# 7. REDUCER
# ==========================================
def reducer_node(state: State) -> dict:
    print("--- Reducer Node ---")
    plan = state["plan"]
    title = plan.get("blog_title", "Blog Post")
    
    ordered = [text for _, text in sorted(state["sections"], key=lambda x: x[0])]
    final_md = f"# {title}\n\n" + "\n\n".join(ordered)
    
    safe_title = re.sub(r"[^a-zA-Z0-9]", "_", title)
    filename = f"{safe_title}.md"
    try:
        Path(filename).write_text(final_md, encoding="utf-8")
        print(f"SUCCESS: Saved to {filename}")
    except Exception as e:
        print(f"Error saving file: {e}")
    
    return {"final": final_md}

# ==========================================
# 8. BUILD GRAPH
# ==========================================
g = StateGraph(State)
g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_node)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

# ==========================================
# 9. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting Blog Generator...")
    try:
        app.invoke({
            "topic": "Microservices vs Monoliths in 2025", 
            "sections": [],
            "evidence": [] 
        })
        print("Done.")
    except Exception as e:
        print(f"Fatal Execution Error: {e}")

Starting Blog Generator...
--- Router Node (Topic: Microservices vs Monoliths in 2025) ---
Router Error: Expecting value: line 1 column 1 (char 0). Defaulting to Hybrid Research.
  > Searching Tavily: Microservices vs Monoliths in 2025 trends 2025
  > Searching Tavily: Microservices vs Monoliths in 2025 best practices
  > Synthesizing evidence...
  > Found 3 valid evidence items.
--- Orchestrator Node ---
Writing Section: Introduction: The Evolving Architectural Landscape
Writing Section: The Case for Microservices in 2025
Writing Section: The Enduring Strengths of the Monolith
Writing Section: The Migration Decision: A 2025 Refactoring Guide
Writing Section: Conclusion: Choosing Your Path Forward
--- Reducer Node ---
SUCCESS: Saved to Microservices_vs_Monoliths_in_2025__A_Practical_Guide_to_Architecture_Choices.md
Done.
